# Compound Triage: Descriptors, ADMET & QM Interaction Energy

**BioPipelines example** — a compute-cheap-to-expensive screening funnel. A mixed library (real Abl kinase inhibitors plus deliberately poor decoys) is filtered on RDKit descriptors and ADMET-AI absorption first; only the drug-like survivors reach Boltz2 co-folding against the Abl kinase domain, and xTB then computes a semi-empirical QM interaction energy on the binding pocket as a physics cross-check on the learned affinity.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import RDKit, ADMETAI, Boltz2, XTB

with Pipeline(project="Setup", job="InstallTools"):
    RDKit.install()
    ADMETAI.install()
    Boltz2.install()
    XTB.install()

## Cell 3: Compound Triage Pipeline

Screens a mixed library against the **Abl kinase domain** (the real target of these inhibitors), spending cheap compute before expensive compute:

1. **RDKit** computes physicochemical descriptors (MW, logP, TPSA, H-bond donors).
2. **ADMET-AI** predicts ~40 ADMET endpoints, including human intestinal absorption (`HIA_Hou`).
3. A **Panda** filter (`MolLogP < 5 and HIA_Hou > 0.8`) drops the non-drug-like decoys — so only chemically and pharmacologically sensible compounds reach the GPU.
4. **Boltz2** co-folds the survivors with the Abl kinase domain and predicts affinity.
5. **DistanceSelector** marks the residues *beyond* 6 Å of the bound ligand, and **PDB.remove** trims each complex down to the binding pocket (keeping the ligand HETATM).
6. **xTB** runs GFN2-xTB on the pocket-sized complex and reports the protein–ligand interaction energy (`e_interaction_kj`) — a physics cross-check, kept tractable by the pocket trim (a full-protein GFN2 single point would take >1 h).

The funnel keeps the expensive Boltz2/xTB steps focused on the survivors, and the QM step on the pocket rather than the whole protein.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines import (PDB, CompoundLibrary, RDKit, ADMETAI, Boltz2,
                          DistanceSelector, XTB, Panda)

with Pipeline(project="KinaseLibrary", job="ADMETTriage"):
    Resources(gpu="A100", time="6:00:00", memory="16GB")

    # Real Abl inhibitors + two deliberately poor decoys that the filter should reject.
    library = CompoundLibrary({
        "dasatinib":  "Cc1nc(Nc2ncc(s2)C(=O)Nc2c(C)cccc2Cl)cc(n1)N1CCN(CCO)CC1",
        "imatinib":   "Cc1ccc(cc1Nc1nccc(n1)-c1cccnc1)NC(=O)c1ccc(cc1)CN1CCN(C)CC1",
        "nilotinib":  "Cc1cn(cn1)-c1cc(cc(c1)C(F)(F)F)NC(=O)c1ccc(C)c(c1)Nc1nccc(n1)-c1cccnc1",
        "bosutinib":  "COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C#N)c(Cl)cc1Cl",
        "greasy_decoy":  "CCCCCCCCCCCCCCCCCCCCCCCCCCCCCC",            # C30 alkane: logP >> 5
        "polar_decoy":   "OCC(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)CO",  # sugar-alcohol: poor HIA
    })

    # Cheap filters first: drop non-drug-like / poorly-absorbed compounds.
    descriptors = RDKit(compounds=library, descriptors=["MolWt", "MolLogP", "TPSA", "NumHDonors"])
    admet = ADMETAI(compounds=library)
    survivors = Panda(tables=[descriptors.tables.descriptors, admet.tables.admet],
                      operations=[Panda.merge(),
                                  Panda.filter("MolLogP < 5 and HIA_Hou > 0.8")],
                      pool=library)

    # Expensive steps only on survivors. Abl kinase domain = the inhibitors' real target.
    abl = PDB("2HYY", chain="A")
    complexes = Boltz2(proteins=abl, ligands=survivors, affinity=True)

    # Trim each complex to the binding pocket so the QM step is tractable.
    pocket = DistanceSelector(structures=complexes, ligand=complexes, distance=6.0)
    trimmed = PDB(complexes, PDB.remove(pocket.tables.selections.beyond, remove_hetatm=False))

    # Semi-empirical QM interaction energy on the pocket-sized complex.
    qm = XTB(structures=trimmed, ligand=complexes, method="gfn2")
qm.tables.interaction_energies